# 🕵️‍♂️ Global Syndicate Taskforce: RL Training Loop

**Training a model to overcome 'Collective Delusion' using OpenEnv, TRL (GRPO), and Unsloth.**

This notebook represents the core pipeline of our project. Current LLMs suffer from 'sycophancy'—if another agent tells them something, they believe it. In our Anti-Money Laundering environment, an LLM must learn to stop trusting hallucinating Tier 1 Analysts, interrogate Bank Liaisons with correct policy mandates, and construct an airtight evidence chain for the Legal Officer.

### 🏆 Our Stack:
- **Environment:** OpenEnv (FastAPI)
- **Base Model:** `unsloth/Qwen2.5-3B-Instruct`
- **RL Framework:** TRL (GRPO - Group Relative Policy Optimization)
- **Compiler:** Unsloth (for extremely fast, memory-efficient LoRA adapters)

## ⚙️ 1. Environment & Dependency Setup
First, we clone our repository directly into the Colab environment and install the required dependencies. We use the `unsloth[colab-new]` package to ensure we get the latest optimizations for GPU execution.

In [7]:
!git clone https://github.com/mdkamranalam/global-syndicate-taskforce.git
%cd global-syndicate-taskforce

# Install Unsloth and standard ML packages
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets requests openenv-core fastapi uvicorn unsloth mergekit llm-blender weave

Cloning into 'global-syndicate-taskforce'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 34 (delta 11), reused 31 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 193.53 KiB | 2.30 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/Users/md.kamranalam/Programming/hackathon/global_syndicate_taskforce_env/notebooks/global-syndicate-taskforce/global-syndicate-taskforce/global-syndicate-taskforce
  Cloning https://github.com/unslothai/unsloth.git to /private/var/folders/s_/7xt2j29s0_dcz28fxwwpnfb00000gn/T/pip-install-ms4sy2pz/unsloth_b38a962357dd4d4097af46928d5634a0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /private/var/folders/s_/7xt2j29s0_dcz28fxwwpnfb00000gn/T/pip-install-ms4sy2pz/unsloth_b38a962357dd4d4097af46928d5634a0
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e13

## 🌍 2. Booting the OpenEnv API Server
Our environment is completely decoupled from the training script. It runs as a stateless REST API (FastAPI) which prevents the LLM from cheating or accessing hidden variables.

We start the `uvicorn` server in the background of this Colab instance so our training loop can ping it.

In [8]:
import subprocess
import time
import requests

# 1. Kill any existing zombie servers
!pkill -f uvicorn

# 2. Start the server using the absolute python path (much safer in Colab)
import sys
print("Starting OpenEnv server...")
server_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000"])

# 3. Wait 5 seconds for boot
time.sleep(5)

# 4. Test the API connection!
try:
    print("Testing connection...")
    response = requests.post("http://localhost:8000/step", 
                             json={"action": {"target_actor": "Tier_1_Analyst", "operation": "fetch_triage_report"}}, 
                             timeout=5)
    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
    if response.status_code == 200:
        print("✅ SUCCESS! The server is alive. You can now run the training script!")
    else:
        print("❌ FAILED! The server is running but rejected the action schema.")
except Exception as e:
    print(f"❌ CRITICAL FAILURE! The server is dead. Error: {e}")


Starting OpenEnv server...


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [50237]
INFO:     Started server process [50329]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Testing connection...
INFO:     127.0.0.1:53934 - "POST /step HTTP/1.1" 200 OK
Status Code: 200
Response: {'observation': {'transaction_id': 'TX-1024', 'amount': 1200.0, 'verified_facts': [], 'system_alerts': 'Analyst Report: Transaction appears STOLEN_IDENTITY.'}, 'reward': -0.05, 'done': False}
✅ SUCCESS! The server is alive. You can now run the training script!


## 🧠 3. Execute the GRPO Training Loop
This is where the magic happens. The `train_grpo_unsloth.py` script will load our base model, generate multiple completions (actions) per prompt, and pass them to our FastAPI environment.

The environment will calculate the **Dense Reward**:
- Falling for the Analyst's hallucination = **-1.00**
- Wasting a step = **-0.05**
- Providing the correct mandate to the Liaison (Theory-of-Mind) = **+0.30**
- Successfully solving the case = **+0.70**

Over 100 steps, you will see the model's reward curve skyrocket as it learns *Institutional Skepticism*.

In [9]:
!python train_grpo_unsloth.py

Traceback (most recent call last):
  File "/Users/md.kamranalam/Programming/hackathon/global_syndicate_taskforce_env/notebooks/global-syndicate-taskforce/global-syndicate-taskforce/global-syndicate-taskforce/train_grpo_unsloth.py", line 1, in <module>
    from unsloth import FastLanguageModel
ModuleNotFoundError: No module named 'unsloth'


## 💾 4. Export the Trained Model
Once training is complete, the LoRA adapters are saved in the `grpo_trained_auditor` directory. We zip these up so they can be downloaded and used for inference in our Hugging Face Space.

In [4]:
import shutil
from google.colab import files

# Zip the trained adapters
shutil.make_archive('grpo_trained_auditor', 'zip', 'grpo_trained_auditor')

# Download the file to your local machine
files.download('grpo_trained_auditor.zip')

ModuleNotFoundError: No module named 'google'